In [ ]:
import os 
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from IPython.core.interactiveshell import InteractiveShell  
from ipywidgets import interact  
InteractiveShell.ast_node_interactivity = "all"   # type: ignore
import ipywidgets as widgets
from ipywidgets import interact 
from pandas import Index
from typing import Any, TypedDict
import torch

In [ ]:
""" 
Required classes. 
"""

class PerStepMetricInfo(TypedDict):
    values: np.ndarray
    save_path: Path
    header: list[str]

In [ ]:
"""
Interactive analysis functionality.
"""

# def show_col_summary(column: Any, frame):
#     print(frame[column].describe())

# def show_summary(dfs: list[pd.DataFrame], v_dir: Path):
#     print(f"Results aggregated over {v_dir.stem} in {v_dir.parent.parent.stem}/{v_dir.parent.stem}")
#     for frame in dfs:
#         interact(show_col_summary, frame=frame, column=list(frame.columns))
#         frame 


In [ ]:
""" 
Visualization functionality. 
"""

def viz_sep_col(metric: PerStepMetricInfo, save_fig: bool = True, save_derived_metric: bool = False) -> None:
    """
    For a metric object instance, visualize each col of `values` with its associated string in `header`.
    Internal output is figure with plots per column, header label combo.
    Optionally save each to compressed numpy file with flags (NOT recommended).
    Optionally save figure (Recommended). 
    TODO: But with what path, might as well maybe create separate instance separate from global df, not really. 
    """
    n_cols = metric["values"].shape[1]
    fig, axs = plt.subplots(1, n_cols, sharex=True, sharey=True, figsize=(4 * n_cols, 4))
    fig.subplots_adjust(wspace=0, hspace=0)

    for i, _ in enumerate(metric["header"]):
        col_vals = metric["values"][:,i]
        col_name = metric["header"][i]
        clean_header = col_name.replace('.', '_')
        col_path = Path(metric['save_path'].parent.resolve() / clean_header)
        axs[i].plot(
            col_vals, 
            # linestyle="dashed", 
            # color="black", 
            # marker="o", 
            label=clean_header
        )
        axs[i].legend()

        if save_derived_metric:
            if not Path.exists(col_path):
                col_path.mkdir(parents=True, exist_ok=True)
            np.savez_compressed(file=col_path.resolve(), derived_metric = col_vals)

    if save_fig: 
        save_path = Path(str(metric['save_path'].resolve()) + ".png")
        fig.savefig(save_path, format="png")
        
def viz_all_col(metric: PerStepMetricInfo, save_fig: bool = True, save_derived_metric: bool = True) -> None:
    """
    For a metric object instance, visualize all col of `values` with its associated string in `header`.
    Internal output is figure with 1 plot for all column , labelled by col name in Index.
    Optionally save as compressed numpy file with flags per viz (Recommended).
    Optionally save figure (Recommended) per viz. 
    """
    fig, axs = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(4, 4))
    metric_df = pd.DataFrame(metric["values"], columns=metric['header'])
    metric_df.plot(sharex=True, sharey=True, ax=axs)

    if save_derived_metric:
        np.savez_compressed(file=metric['save_path'].resolve(), header = list(metric_df.columns), values = metric_df.values)

    if save_fig: 
        save_path = Path(str(metric['save_path'].resolve()) + ".png")
        fig.savefig(save_path.resolve(), format="png")

In [ ]:
"""
Standard metric analysis helpers. 
"""
def analyse_accuracy_metrics(metrics_paths: list[Path], prefix:str = "standard"):
    """ 
    Equivalent func as `analyse_standard_metrics` except a prefix is used to 
    further denote the naming used for saved metrics in the same directory, 
    between GCS and RGCS. 

    This is kept until possible differentiating functionality is required, 
    otherwise TODO: Deprecate.
    """
    num_metrics = len(metrics_paths)

    # Sizes are unknown unless peeking is performed
    sum = np.zeros(0)
    sum_sq = np.zeros(0)
    header = [""]
    # Save initial path
    enumerated_metrics = enumerate(metrics_paths)

    for i, train_metrics_path in enumerated_metrics:
        metrics = np.load(train_metrics_path.resolve(), allow_pickle=True)
        
        if i == 0:
            sum = np.zeros_like(metrics)
            sum_sq = np.zeros_like(metrics)
            header = metrics.dtype.names

        sum = np.sum([sum, metrics], axis=(0))
        sum_sq = np.sum([sum_sq, np.square(metrics)], axis=(0))

    global_mean = sum / num_metrics
    global_var = (sum_sq / num_metrics) - (np.square(global_mean))
    global_std = np.sqrt(global_var)
    dir = list(metrics_paths)[0].parent
    summary_metrics = [
        PerStepMetricInfo(
            values=global_std,
            save_path=dir / f"{prefix}_global_std",
            header=["global_std_" + h for h in header]
        ),
        PerStepMetricInfo(
            values=global_mean,
            save_path= dir/ f"{prefix}_global_mean",
            header=["global_mean_" + h for h in header]
        ),
    ]

    for i, summary_metric in enumerate(summary_metrics):
        print(f"Standard summary metric immediate file results and subdirs all saved into {summary_metric['save_path'].resolve()}")
        viz_sep_col(summary_metric, save_derived_metric=True)
        viz_all_col(summary_metric)

In [ ]:
""" 
GCS analysis helpers. 
"""
def analyse_cs_metrics(metrics_paths: list[Path], prefix:str) -> None:
    """ 
    Perform analysis steps of correlation store metrics. 
    """
    analyse_accuracy_metrics(metrics_paths, prefix)

In [ ]:
outputs_dir = Path().resolve().parent / "outputs"
corr_store_names = dict(normed="GCS", raw="RGCS")

In [ ]:
""" 
Main entry point for analysis. 
TODO: 
- Fix FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\motlh\\coder_life\\COS700\\outputs\\adam_class_imbalance_ratios\\multijob\\v_0\\run_0\\train\\gcs_global_std\\global_stdconv1.weight.npz'
via change .weight. This is in viz_sep_col, trying for first time. 
- Merge analyze like with analyze, to not duplicate code changes. Can try decorator maybe.
- Change to global/std/"<old>" naming for graphs. 
- Add cleaning routine (redundant if training works fine and I know consistent output directories)
    - So cleaning is just for current mistakes of files added, or simply retrain. 
- Debug save as png does not work unless explicit. 
- Save GCS as npz with metadata e.g. step count, header, stitched after all epochs, not saved per epoch, just like normal metrics, as flat file. 
- Save metrics as npz with meta data e.g. step count with callbacks likely. 
- Run to check efficiency, here then cluster. 
"""

plt.switch_backend('WebAgg')

for experim_dir in sorted(outputs_dir.iterdir()):
    version_dirs = sorted(experim_dir.glob("multijob/R*"))

    for v_dir in version_dirs:
        # Get all csv file path matches to default logger file name for standard metrics
        train_acc_paths = sorted(v_dir.rglob("run*/train/train_acc.pt", case_sensitive=False))
        analyse_accuracy_metrics(train_acc_paths)

        val_acc_paths = sorted(v_dir.rglob("run*/train/val_acc.pt", case_sensitive=False))
        analyse_accuracy_metrics(val_acc_paths)

        # Get all csv file path matches to custom name for both cs metrics
        gcs_paths = sorted(v_dir.rglob(f"run*/train/*{corr_store_names["normed"]}*.pt", case_sensitive=False))
        analyse_cs_metrics(gcs_paths, "gcs")

        rgcs_paths = sorted(v_dir.rglob(f"run*/train/*{corr_store_names["raw"]}*.pt", case_sensitive=False))
        analyse_cs_metrics(rgcs_paths, "rgcs")

plt.show()